In [10]:
import torch
import torch.nn as nn
from torch import tensor 

### Without

In [75]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc0 = nn.Linear(1, 1, bias=False)
        self.fc1 = nn.Linear(1, 1, bias=False)
        self.fc2 = nn.Linear(1, 1, bias=False)
        self.fc3 = nn.Linear(1, 1, bias=False)
        self.fc4 = nn.Linear(1, 1, bias=False)
        self.fc5 = nn.Linear(1, 1, bias=False)
    
    def forward(self, x):
        x = self.fc0(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        x = self.fc4(x)
        x = self.fc5(x)
        return x 

In [76]:
state_dict = {
    "fc0.weight": torch.tensor([[0.1]]),
    "fc1.weight": torch.tensor([[0.2]]),
    "fc2.weight": torch.tensor([[0.3]]),
    "fc3.weight": torch.tensor([[0.4]]),
    "fc4.weight": torch.tensor([[0.5]]),
    "fc5.weight": torch.tensor([[0.6]]),
}
model = MyModel()
model.load_state_dict(state_dict)

<All keys matched successfully>

In [78]:
x = tensor([0.7])
y_pred = model(x)
y_pred.item()

0.0005040000542066991

In [79]:
loss_fn = nn.MSELoss()
loss = loss_fn(y_pred, y_target)
loss.backward()

In [80]:
model.fc0.weight.grad

tensor([[-0.0030]])

In [81]:
x = tensor(0.7)
w0 = tensor(0.1)
w1 = tensor(0.2)
w2 = tensor(0.3)
w3 = tensor(0.4)
w4 = tensor(0.5)
w5 = tensor(0.6)

y_pred = (x * w0 * w1 * w2 * w3 * w4 * w5)
y_pred

tensor(0.0005)

In [82]:
dL_ypred = 2 * (y_pred.item() - y_target.item())
dypred_dw0 = (x * w1 * w2 * w3 * w4 * w5)
dL_dw0 = dL_ypred * dypred_dw0
dL_dw0

tensor(-0.0030)

### With Residual Connection

In [60]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc0 = nn.Linear(1, 1, bias=False)
        self.fc1 = nn.Linear(1, 1, bias=False)
        self.fc2 = nn.Linear(1, 1, bias=False)
        self.fc3 = nn.Linear(1, 1, bias=False)
        self.fc4 = nn.Linear(1, 1, bias=False)
        self.fc5 = nn.Linear(1, 1, bias=False)
    
    def forward(self, x):
        residual = x
        x = self.fc0(x)
        x = self.fc1(x)
        x = self.fc2(x)
    
        x = x + residual
        residual = x
        x = self.fc3(x)
        x = self.fc4(x)
        x = self.fc5(x)
        x = x + residual
    
        return x 

In [61]:
state_dict = {
    "fc0.weight": torch.tensor([[0.1]]),
    "fc1.weight": torch.tensor([[0.2]]),
    "fc2.weight": torch.tensor([[0.3]]),
    "fc3.weight": torch.tensor([[0.4]]),
    "fc4.weight": torch.tensor([[0.5]]),
    "fc5.weight": torch.tensor([[0.6]]),
}
model = MyModel()
model.load_state_dict(state_dict)

<All keys matched successfully>

In [62]:
x = tensor([0.7])

In [63]:
y_pred = model(x)

In [64]:
y_pred.item()

0.788703978061676

In [65]:
y_target = tensor([0.3])

In [66]:
(0.7 * 0.1 * 0.2 * 0.3 + 0.7) * 0.4 * 0.5 * 0.6 + (0.7 * 0.1 * 0.2 * 0.3 + 0.7)

0.788704

In [67]:
loss_fn = nn.MSELoss()

In [68]:
loss = loss_fn(y_pred, y_target)

In [69]:
loss.backward()

In [70]:
model.fc0.weight.grad

tensor([[0.0460]])

In [71]:
model.fc1.weight.grad

tensor([[0.0230]])

In [72]:
(0.7 * 0.1 * 0.2 * 0.3 + 0.7) * 0.4 * 0.5 * 0.6 + (0.7 * 0.1 * 0.2 * 0.3 + 0.7)

x = tensor(0.7)
w0 = tensor(0.1)
w1 = tensor(0.2)
w2 = tensor(0.3)
w3 = tensor(0.4)
w4 = tensor(0.5)
w5 = tensor(0.6)

y_pred = (x * w0 * w1 * w2 + x) * w3 * w4 * w5 + (x * w0 * w1 * w2 + x)
y_pred

tensor(0.7887)

In [73]:
y_pred = (x * w0 * w1 * w2 * w3 * w4 * w5) + (x * w3 * w4 * w5) + (x * w0 * w1 * w2) + (x)
y_pred

tensor(0.7887)

In [74]:
dL_ypred = 2 * (y_pred.item() - y_target.item())
dypred_dw0 = (x * w1 * w2 * w3 * w4 * w5) + (x * w1 * w2)
dL_dw0 = dL_ypred * dypred_dw0
dL_dw0

tensor(0.0460)